> 在线讲义：[第 22 章 · 输入 embedding 与输出 softmax：词怎么变成向量，向量怎么变回词](https://wikieden.github.io/transformer-visual/lessons/22-embedding-softmax.html) · 本书《从零读懂 Transformer》

# 第 22 章 · 输入 embedding 与输出 softmax：词怎么变成向量，向量怎么变回词

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/22-embedding-softmax.html) · 仓库里 `transformer-deep-dive/notes/lessons/22-embedding-softmax.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 第一步 查表 | $\mathbf{x} = E[i]$ | `emb = nn.Embedding(V, d)`；`emb(ids)` | "狗 咬 人" → 编号 [1, 2, 3] → 3 × 4 |
| 数学原理 one-hot | one-hot × E | `one_hot(ids, V).float() @ emb.weight` | 和 `emb(ids)` 完全一样 |
| 第一步 意思相近 | 点积 | `E[1] @ E[3]`、`E[1] @ E[2]` | 狗·人 = 0.89，狗·咬 = −0.62 |
| 第三步 共用一张表 | $W_{out} = E^\top$ | `lm_head = nn.Linear(d, V, bias=False)`；`lm_head.weight = emb.weight` | `lm_head.weight is emb.weight` → True |
| 第二步 打分 + softmax | $\text{logits} = hW_{out}$，$p = \text{softmax}$ | `logits = lm_head(h)`；`p = torch.softmax(logits, dim=-1)` | bites 2.9 分、0.537；加起来 1.0 |
| 第二步 greedy | 挑最大 | `p.argmax()` | 6 → bites |
| 第四步 乘 √d 加位置 | $\sqrt{d_{model}}\,E[i] + PE_{pos}$ | `math.sqrt(d) * emb(ids) + pe` | 狗坐 0 号座：[1.8, −0.6, 0.8, 0.2] + [0, 1, 0, 1] = [1.8, 0.4, 0.8, 1.2] |
| 第四步 为什么乘 | 音量 | `torch.randn(512) / math.sqrt(512)` | 表 E 的数平均 ≈ 0.035；乘 √512 后 ≈ 0.80；位置编码 ≈ 0.58 |

表里的数和讲义一样，都是编的：8 个词，每个词 4 个数（论文 V ≈ 37000、d_model = 512）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import math
import torch
from torch import nn
WORDS = ["<开始>", "狗", "咬", "人", "<结束>", "dog", "bites", "man"]
D_MODEL = 4
E_ROWS = [
    [0.1, 0.0, -0.2, 0.3],  # 0 <开始>
    [0.9, -0.3, 0.4, 0.1],  # 1 狗
    [-0.5, 0.8, 0.2, -0.1],  # 2 咬
    [0.7, -0.2, 0.5, 0.0],  # 3 人
    [0.0, 0.2, -0.4, 0.6],  # 4 <结束>
    [0.8, -0.4, 0.3, 0.2],  # 5 dog
    [-0.3, 0.7, -0.2, 0.5],  # 6 bites
    [0.6, -0.1, 0.6, -0.1],  # 7 man
]
H = [-1.0, 2.0, -1.0, 2.0]

**函数 `make_embedding`**：第一步：表 E = nn.Embedding(V, d_model)，权重换成讲义里那张表。

In [ ]:
def make_embedding() -> nn.Embedding:
    """第一步：表 E = nn.Embedding(V, d_model)，权重换成讲义里那张表。"""
    emb = nn.Embedding(len(WORDS), D_MODEL)
    with torch.no_grad():
        emb.weight.copy_(torch.tensor(E_ROWS))
    return emb

**函数 `tokenize`**：① 词 → 编号。真正的 tokenizer（BPE）在 CS336 a1 写，这里按空格切、查名单。

In [ ]:
def tokenize(sentence: str) -> torch.Tensor:
    """① 词 → 编号。真正的 tokenizer（BPE）在 CS336 a1 写，这里按空格切、查名单。"""
    return torch.tensor([WORDS.index(w) for w in sentence.split()])

**函数 `lookup`**：② 编号 → 向量：x = E[i]，就是取第 i 行。

In [ ]:
def lookup(emb: nn.Embedding, ids: torch.Tensor) -> torch.Tensor:
    """② 编号 → 向量：x = E[i]，就是取第 i 行。"""
    return emb(ids)

**函数 `lookup_by_one_hot`**：附录"数学原理"：查表 = one-hot 乘矩阵。结果和 emb(ids) 一样，只是白做了很多乘 0。

In [ ]:
def lookup_by_one_hot(emb: nn.Embedding, ids: torch.Tensor) -> torch.Tensor:
    """附录"数学原理"：查表 = one-hot 乘矩阵。结果和 emb(ids) 一样，只是白做了很多乘 0。"""
    one_hot = nn.functional.one_hot(ids, num_classes=len(WORDS)).float()  # 几个词 × V
    return one_hot @ emb.weight  # (几个词 × V) @ (V × d) = 几个词 × d

**函数 `tied_lm_head`**：第三步：输出层和 embedding 共用同一张表。

In [ ]:
def tied_lm_head(emb: nn.Embedding) -> nn.Linear:
    """第三步：输出层和 embedding 共用同一张表。

    nn.Linear(d, V) 的 weight 形状是 V × d（PyTorch 存的是"输出 × 输入"），算的是 h @ weight.T。
    它和 embedding 表 E 的形状正好一样，所以直接让两者指向同一个参数，不用转置。
    """
    lm_head = nn.Linear(D_MODEL, len(WORDS), bias=False)
    lm_head.weight = emb.weight
    return lm_head

**函数 `next_token`**：第二步：logits = h W_out，p = softmax(logits)，greedy 挑最大的。

In [ ]:
@torch.no_grad()  # 生成时只用不学，不需要记梯度
def next_token(lm_head: nn.Linear, h: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, int]:
    """第二步：logits = h W_out，p = softmax(logits)，greedy 挑最大的。"""
    logits = lm_head(h)
    p = torch.softmax(logits, dim=-1)
    return logits, p, int(p.argmax())

**函数 `sinusoidal_pe`**：第 16 章的位置编码：第 2i 位放 sin，第 2i+1 位放 cos。

In [ ]:
def sinusoidal_pe(pos: int, d: int) -> torch.Tensor:
    """第 16 章的位置编码：第 2i 位放 sin，第 2i+1 位放 cos。"""
    pe = torch.zeros(d)
    for i in range(d // 2):
        angle = pos / 10000 ** (2 * i / d)
        pe[2 * i] = math.sin(angle)
        pe[2 * i + 1] = math.cos(angle)
    return pe

**函数 `embed_input`**：第四步：x = sqrt(d_model) · E[i] + PE_pos，这才是进第 1 层的东西。

In [ ]:
def embed_input(emb: nn.Embedding, ids: torch.Tensor) -> torch.Tensor:
    """第四步：x = sqrt(d_model) · E[i] + PE_pos，这才是进第 1 层的东西。"""
    pe = torch.stack([sinusoidal_pe(pos, D_MODEL) for pos in range(len(ids))])
    return math.sqrt(D_MODEL) * emb(ids) + pe

**函数 `show`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def show(t: torch.Tensor, digits: int = 3) -> str:
    return str([round(float(v), digits) for v in t.detach().flatten()])

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
torch.set_printoptions(precision=3, sci_mode=False)
emb = make_embedding()
print("== 第一步：查表 ==")
ids = tokenize("狗 咬 人")
x = lookup(emb, ids)
print("编号:", ids.tolist())
print("形状:", tuple(x.shape), "  （行数 = 句子里几个词，列数 = d_model）")
for w, row in zip(["狗", "咬", "人"], x):
    print(f"  {w}: {show(row, 1)}")
print("one-hot 乘矩阵结果一样:", torch.allclose(x, lookup_by_one_hot(emb, ids)))
dog, bite, man = emb.weight.detach()[1:4]
print(f"狗·人 = {float(dog @ man):.2f}   狗·咬 = {float(dog @ bite):.2f}")
print("表 E 会被训练调整:", emb.weight.requires_grad)

In [ ]:
print("\n== 第二、三步：输出打分 + softmax（共用表 E）==")
lm_head = tied_lm_head(emb)
print("输出层和 embedding 是同一个参数:", lm_head.weight is emb.weight)
logits, p, best = next_token(lm_head, torch.tensor(H))
for w, s, q in zip(WORDS, logits, p):
    print(f"  {w:<6} logit {float(s):5.2f}   概率 {float(q):.3f}")
print("概率加起来:", round(float(p.sum()), 6), "  greedy 选:", WORDS[best])

In [ ]:
print("\n== 第四步：乘 sqrt(d_model) 再加位置编码 ==")
print("√d_model =", math.sqrt(D_MODEL))
print("狗 ×√d:", show(math.sqrt(D_MODEL) * emb.weight[1], 1))
print("座位 0 的位置编码:", show(sinusoidal_pe(0, D_MODEL), 1))
print("进第 1 层的 3 行:")
for w, row in zip(["狗", "咬", "人"], embed_input(emb, ids)):
    print(f"  {w}: {show(row)}")

In [ ]:
print("\n== 第四步补充：为什么要乘（d_model = 512 时的音量）==")
torch.manual_seed(0)
d = 512
e_row = torch.randn(d) / math.sqrt(d)  # 共用的表一般用很小的随机数初始化
print(f"表 E 一行的数，平均大小 ≈ {float(e_row.abs().mean()):.3f}")
print(f"乘 √512 ≈ {math.sqrt(d):.1f} 以后 ≈ {float((math.sqrt(d) * e_row).abs().mean()):.2f}")
pe = torch.stack([sinusoidal_pe(pos, d) for pos in range(50)])
print(f"位置编码的数，平均大小 ≈ {float(pe.abs().mean()):.2f}")

## 3. 代码里才看得到的

1. **`nn.Linear` 的 weight 是"输出 × 输入"**：`nn.Linear(4, 8)` 的 weight 形状是 8 × 4，算的是 `h @ weight.T`。它和表 E（V × d）形状一模一样，所以共用时直接 `lm_head.weight = emb.weight`，**不用自己转置**；讲义里的 $E^\top$ 藏在 `nn.Linear` 里面了。nn = neural network（神经 / 网络），PyTorch 放各种"层"的地方。
2. **共用是"同一个参数"，不是复制一份**：`is` 判断为 True。训练时输入那头和输出那头调的是同一块内存，这就是第三步说的"两头都在调同一行"。
3. **表 E 会被训练**：`emb.weight.requires_grad` 是 True（requires gradient：需要 / 梯度，意思是训练时要算"往哪挪"，续篇第 2 章讲）。生成时只用不学，代码里用 `@torch.no_grad()` 关掉。

## 4. 自己改一改

- 把 `H` 改成 [1, −0.5, 0.5, 0]，看 greedy 选出哪个词；它和表里哪一行最像？
- 把 `H` 每个数都乘 3，再跑：最大的概率变大了还是变小了？（对照动画里"h 的长度"那个滑块）

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 词先换编号，编号查一行；表是学出来的，原文译文都查它。
> 最后一行去打分，词表每个词一分；softmax 分比例，挑最大或束搜索。
> 一本字典两头用：正查取向量，反查比相像。
> 查完乘根号 d，词和座位一样响。